# Benefits Market Intelligence Exploratory Data Analysis: Form 5500 - Schedule C Part 1, Item 2

## Libraries

In [ ]:
# Libraries
import pandas as pd
import duckdb
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from benefits_market_intelligence.config.paths import DB_PATH, FIGURES_PATH, EDA_PATH
from benefits_market_intelligence.visualizations.save import save_figure

In [ ]:
# Displaying all
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## Load Form 5500 - Schedule C Part 1, Item 2 data

In [ ]:
# Loading data
with duckdb.connect(DB_PATH, read_only=True) as con:
    SCH_C_P1_I2 = con.sql("SELECT * FROM silver.SCH_C_P1_I2").df()

In [ ]:
# Convert pandas-made object columns to string
obj_cols = SCH_C_P1_I2.dtypes[SCH_C_P1_I2.dtypes == "object"].index
SCH_C_P1_I2[obj_cols] = SCH_C_P1_I2[obj_cols].astype("str")

## Viewing data

In [ ]:
# Head of SCH_C_P1_I2
SCH_C_P1_I2.head()

## Shape of data

In [ ]:
# Shape
print(f"Rows: {SCH_C_P1_I2.shape[0]}\nColumns: {SCH_C_P1_I2.shape[1]}")

## Data types

In [ ]:
# Data type counts
SCH_C_P1_I2.dtypes.value_counts()

## Any missing data?

In [ ]:
# Columns with no missing data
SCH_C_P1_I2.isna().sum()[SCH_C_P1_I2.isna().sum() == 0]

In [ ]:
# Missing data
pd.DataFrame(
    {
        "missing_count": SCH_C_P1_I2.isna().sum(),
        "missing_percent": SCH_C_P1_I2.isna().mean().mul(100),
    }
).query("missing_count > 0").sort_values(
    "missing_percent", ascending=False
).rename_axis("column_name").reset_index()

## Plotly setup

In [ ]:
# Create paths
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
EDA_PATH.mkdir(parents=True, exist_ok=True)

## Prepare data for analysis

In [ ]:
# Prepare data for analysis
numeric_cols = ["PROVIDER_OTHER_DIRECT_COMP_AMT", "PROV_OTHER_TOT_IND_COMP_AMT"]
for c in numeric_cols:
    SCH_C_P1_I2[c] = pd.to_numeric(SCH_C_P1_I2[c], errors="coerce")

# Clean up the role/relation text field
SCH_C_P1_I2["PROVIDER_OTHER_RELATION"] = (
    SCH_C_P1_I2["PROVIDER_OTHER_RELATION"]
    .replace("nan", np.nan)
    .str.strip()
    .str.upper()
)

# Total compensation (direct + indirect), treating missing as 0 where at
# least one of the two amounts is present
SCH_C_P1_I2["TOTAL_COMP_AMT"] = SCH_C_P1_I2["PROVIDER_OTHER_DIRECT_COMP_AMT"].fillna(
    0
) + SCH_C_P1_I2["PROV_OTHER_TOT_IND_COMP_AMT"].fillna(0)

# Clean state field for the geographic view
SCH_C_P1_I2["PROVIDER_OTHER_US_STATE"] = (
    SCH_C_P1_I2["PROVIDER_OTHER_US_STATE"]
    .replace("nan", np.nan)
    .str.strip()
    .str.upper()
)

## Most common service provider roles

In [ ]:
# Most common service provider roles
role_counts = (
    SCH_C_P1_I2["PROVIDER_OTHER_RELATION"]
    .dropna()
    .value_counts()
    .head(15)
    .sort_values(ascending=True)
)

if len(role_counts):
    fig = px.bar(
        x=role_counts.values,
        y=role_counts.index,
        orientation="h",
        title="Most Common Service Provider Roles (Schedule C, Part 1, Item 2)",
        labels={"x": "Number of provider records", "y": "Provider role"},
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    save_figure(fig, "provider_role_distribution", EDA_PATH)
    fig.show()
else:
    print("Skipped: no provider role values found.")

## Total direct compensation by provider role

In [ ]:
# Total direct compensation by provider role
comp_by_role = (
    SCH_C_P1_I2.dropna(subset=["PROVIDER_OTHER_RELATION"])
    .groupby("PROVIDER_OTHER_RELATION")["PROVIDER_OTHER_DIRECT_COMP_AMT"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .sort_values(ascending=True)
)

if len(comp_by_role):
    fig = px.bar(
        x=comp_by_role.values,
        y=comp_by_role.index,
        orientation="h",
        title="Total Direct Compensation by Service Provider Role",
        labels={"x": "Total direct compensation ($)", "y": "Provider role"},
    )
    fig.update_xaxes(tickprefix="$", separatethousands=True)
    save_figure(fig, "total_direct_compensation_by_role", EDA_PATH)
    fig.show()
else:
    print("Skipped: no direct compensation values found.")

## Average direct versus indirect compensation by role

In [ ]:
# Average direct versus indirect compensation by role
top_roles = (
    SCH_C_P1_I2["PROVIDER_OTHER_RELATION"].dropna().value_counts().head(10).index
)

avg_comp = (
    SCH_C_P1_I2[SCH_C_P1_I2["PROVIDER_OTHER_RELATION"].isin(top_roles)]
    .groupby("PROVIDER_OTHER_RELATION")[
        ["PROVIDER_OTHER_DIRECT_COMP_AMT", "PROV_OTHER_TOT_IND_COMP_AMT"]
    ]
    .mean()
    .rename(
        columns={
            "PROVIDER_OTHER_DIRECT_COMP_AMT": "Average direct compensation",
            "PROV_OTHER_TOT_IND_COMP_AMT": "Average indirect compensation",
        }
    )
    .sort_values("Average direct compensation", ascending=False)
)

if len(avg_comp):
    avg_comp_long = avg_comp.reset_index().melt(
        id_vars="PROVIDER_OTHER_RELATION",
        var_name="Compensation type",
        value_name="Amount",
    )
    fig = px.bar(
        avg_comp_long,
        x="PROVIDER_OTHER_RELATION",
        y="Amount",
        color="Compensation type",
        barmode="group",
        title="Average Direct vs. Indirect Compensation by Provider Role",
        labels={
            "PROVIDER_OTHER_RELATION": "Provider role",
            "Amount": "Average compensation ($)",
        },
    )
    fig.update_yaxes(tickprefix="$", separatethousands=True)
    fig.update_xaxes(tickangle=35)
    save_figure(fig, "direct_vs_indirect_compensation_by_role", EDA_PATH)
    fig.show()
else:
    print("Skipped: no roles with compensation data found.")

## Most highly compensated service providers

In [ ]:
# Most highly compensated service providers
top_providers = (
    SCH_C_P1_I2.dropna(subset=["PROVIDER_OTHER_NAME"])
    .groupby("PROVIDER_OTHER_NAME")["TOTAL_COMP_AMT"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .sort_values(ascending=True)
)

if len(top_providers):
    fig = px.bar(
        x=top_providers.values,
        y=top_providers.index,
        orientation="h",
        title="Top 20 Service Providers by Total Compensation",
        labels={"x": "Total compensation ($)", "y": "Provider name"},
    )
    fig.update_xaxes(tickprefix="$", separatethousands=True)
    fig.update_layout(height=650)
    save_figure(fig, "top_compensated_providers", EDA_PATH)
    fig.show()
else:
    print("Skipped: no provider names with compensation data found.")

## Compensation trend over time

In [ ]:
# Compensation trend over time
comp_by_year = (
    SCH_C_P1_I2.groupby("FORM_YEAR")[
        ["PROVIDER_OTHER_DIRECT_COMP_AMT", "PROV_OTHER_TOT_IND_COMP_AMT"]
    ]
    .sum()
    .rename(
        columns={
            "PROVIDER_OTHER_DIRECT_COMP_AMT": "Direct compensation",
            "PROV_OTHER_TOT_IND_COMP_AMT": "Indirect compensation",
        }
    )
    .sort_index()
)

if len(comp_by_year):
    comp_by_year_long = comp_by_year.reset_index().melt(
        id_vars="FORM_YEAR", var_name="Compensation type", value_name="Amount"
    )
    fig = px.line(
        comp_by_year_long,
        x="FORM_YEAR",
        y="Amount",
        color="Compensation type",
        markers=True,
        title="Total Service Provider Compensation by Form Year",
        labels={"FORM_YEAR": "Form year", "Amount": "Total compensation ($)"},
    )
    fig.update_yaxes(tickprefix="$", separatethousands=True)
    save_figure(fig, "compensation_trend_by_year", EDA_PATH)
    fig.show()
else:
    print("Skipped: no data available by form year.")

## Geographic distribution of service providers

In [ ]:
# Geographic distribution of service providers
state_counts = (
    SCH_C_P1_I2["PROVIDER_OTHER_US_STATE"]
    .dropna()
    .value_counts()
    .rename_axis("state")
    .reset_index(name="provider_count")
)

# Keep to the 50 states + DC so odd codes don't distort the map
valid_states = {
    "AL",
    "AK",
    "AZ",
    "AR",
    "CA",
    "CO",
    "CT",
    "DE",
    "FL",
    "GA",
    "HI",
    "ID",
    "IL",
    "IN",
    "IA",
    "KS",
    "KY",
    "LA",
    "ME",
    "MD",
    "MA",
    "MI",
    "MN",
    "MS",
    "MO",
    "MT",
    "NE",
    "NV",
    "NH",
    "NJ",
    "NM",
    "NY",
    "NC",
    "ND",
    "OH",
    "OK",
    "OR",
    "PA",
    "RI",
    "SC",
    "SD",
    "TN",
    "TX",
    "UT",
    "VT",
    "VA",
    "WA",
    "WV",
    "WI",
    "WY",
    "DC",
}
state_counts = state_counts[state_counts["state"].isin(valid_states)]

if len(state_counts):
    fig = px.choropleth(
        state_counts,
        locations="state",
        locationmode="USA-states",
        color="provider_count",
        scope="usa",
        color_continuous_scale="Blues",
        title="Geographic Distribution of Service Providers by State",
        labels={"provider_count": "Number of provider records"},
    )
    save_figure(fig, "provider_geographic_distribution", EDA_PATH)
    fig.show()
else:
    print("Skipped: no usable US state values found.")

## Prevalence of indirect compensation by provider role

In [ ]:
# Prevalence of indirect compensation by provider role
ind_comp_map = {"1": "Received indirect compensation", "2": "No indirect compensation"}

tmp = SCH_C_P1_I2[SCH_C_P1_I2["PROVIDER_OTHER_RELATION"].isin(top_roles)].copy()
tmp["Indirect comp status"] = tmp["PROV_OTHER_INDIRECT_COMP_IND"].map(ind_comp_map)
tmp = tmp.dropna(subset=["Indirect comp status"])

if len(tmp):
    share = (
        tmp.groupby(["PROVIDER_OTHER_RELATION", "Indirect comp status"])
        .size()
        .reset_index(name="count")
    )
    share["percent"] = (
        share["count"]
        / share.groupby("PROVIDER_OTHER_RELATION")["count"].transform("sum")
        * 100
    )
    fig = px.bar(
        share,
        x="PROVIDER_OTHER_RELATION",
        y="percent",
        color="Indirect comp status",
        barmode="stack",
        title="Share of Records with Indirect Compensation, by Provider Role",
        labels={
            "PROVIDER_OTHER_RELATION": "Provider role",
            "percent": "Share of records (%)",
        },
    )
    fig.update_xaxes(tickangle=35)
    save_figure(fig, "indirect_compensation_prevalence_by_role", EDA_PATH)
    fig.show()
else:
    print("Skipped: no usable indirect compensation indicator values found.")